# ✏️ Sesión 6 · Práctica 
## Tipos de vino
Reconoce entre **3 tipos de vino** a partir de su análisis químico (13 medidas).

👉 Completa donde pone `# ✏️ TU CÓDIGO AQUÍ`. Copia del cuaderno de ejemplo y cambia lo necesario.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

**1.** Carga los datos y mira las primeras filas.

In [ ]:
vino = load_wine(as_frame=True)
datos = vino.frame
# ✏️ muestra las primeras filas
print(datos.shape)
display(datos.info())
print("target datos: ", datos["target"].unique())
display(datos)

**2.** Separa X (todas las columnas menos `target`) e y (`target`). Después train y test.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
X = datos.loc[:, datos.columns != "target"]
y = datos["target"]

print(f"Dimension de los datos para predecir {X.shape}")
print(f"Dimension de las respuestas {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("-"*15, "-"*15)
print(f"Dimension de datos para entrenar {X_train.shape}")
print(f"Dimension de datos de prueba {X_test.shape}")


**3.** Los números de este CSV son muy distintos (unos valen 0.3 y otros 1000). Los ponemos en la misma escala, como hicimos en scikit-learn.

*(Esta celda ya está hecha.)*

In [ ]:
escala = StandardScaler()
X_train = escala.fit_transform(X_train)
X_test = escala.transform(X_test)

**4.** Crea la red: **13 entradas**, 1 capa oculta de 16 con `relu` y una salida de **3 opciones** con `softmax`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
red = keras.Sequential([
    keras.Input(shape =(13,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])

red.compile(loss="sparse_categorical_crossentropy",
            optimizer=keras.optimizers.Adam(0.01),
            metrics=["accuracy"])

**5.** Entrena 50 épocas y mira el % de aciertos en el examen.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
red.fit(X_train, y_train, epochs=50, verbose=0)

perdida, aciertos = red.evaluate(X_test, y_test, verbose=0)
print("Aciertos en el examen:", round(aciertos * 100), "%")

**6.** ¿Qué tipo de vino es el **primero** del examen? Dibuja los % en una gráfica de barras.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
predicciones = red.predict(X_test, verbose=0)
etiquetas = ["rojo", "blanco", "rosa"]

clases_predichas = np.argmax(predicciones, axis=1)

# Admite y_test como etiquetas (0, 1, 2) o como matriz one-hot
y_real = np.asarray(y_test)
if y_real.ndim == 2 and y_real.shape[1] == len(etiquetas):
    y_real = np.argmax(y_real, axis=1)
else:
    y_real = y_real.ravel().astype(int)

conteo_real = np.bincount(y_real, minlength=len(etiquetas))
conteo_predicho = np.bincount(clases_predichas, minlength=len(etiquetas))

x = np.arange(len(etiquetas))
ancho = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - ancho / 2, conteo_real, ancho, label="Real")
plt.bar(x + ancho / 2, conteo_predicho, ancho, label="Predicho")

plt.xticks(x, etiquetas)
plt.xlabel("Tipo de vino (target)")
plt.ylabel("Número de vinos")
plt.title("Tipos reales y predichos")
plt.legend()
plt.tight_layout()
plt.show()